In [2]:
import requests
import pandas as pd

# Define the GraphQL query and variables
query = """
query Attestations($where: AttestationWhereInput) {
  attestations(where: $where) {
    attester
    data
    decodedDataJson
    expirationTime
    id
    ipfsHash
    isOffchain
    recipient
    refUID
    revocable
    revocationTime
    revoked
    schema {
      schemaNames {
        name
      }
    }
    schemaId
    time
    timeCreated
    txid
  }
}
"""

variables = {
    "where": {
        "schemaId": {
            "equals": "0xef874554718a2afc254b064e5ce9c58c9082fb9f770250499bf406fc112bd315"
        }
    }
}

# Define the URL of the GraphQL endpoint
url = 'https://optimism.easscan.org/graphql'

# Make the request to the GraphQL endpoint
response = requests.post(url, json={'query': query, 'variables': variables})

# Check if the request was successful
if response.status_code == 200:
    data = response.json()['data']['attestations']
    
    # Convert the list of data to a DataFrame
    df = pd.DataFrame(data)
    
    # Normalize nested JSON data (e.g., 'schema.schemaNames')
    schema_names = df['schema'].apply(lambda x: ', '.join([name['name'] for name in x['schemaNames']]) if x['schemaNames'] else '')
    df['schemaNames'] = schema_names
    
    # Drop the original 'schema' column
    df = df.drop(columns=['schema'])
    
    # Save the DataFrame to a CSV file
    df.to_csv('../../data/measuring_the_cop/attestations.csv', index=False)
    print('Data saved to attestations.csv')
else:
    print(f"Query failed to run with a {response.status_code}.")
    print(response.text)


Data saved to attestations.csv
